# Visualize Crime Data with ggplot2
## Solution Notebook — Criminology Priority Analysis

Complete ggplot2 code for every chart. Images shown below each corresponding block (generated to match the R output).

**Flowchart:**

![ggplot2 Pipeline](criminology_ggplot2_flowchart.png)


## 0. Setup — Load libraries

In [ ]:
library(ggplot2)
library(dplyr)
library(readr)
# All required packages loaded.


## 1. Load and prepare data

In [ ]:
crimes <- read_csv("data/crimes.csv")
crimes <- crimes %>%
  mutate(
    solved_label = ifelse(solved, "Solved", "Unsolved"),
    priority = ifelse(monthly_incidents > 15000 & type != "Theft", "High Priority", "Other")
  )
head(crimes)
# Expected: 40 rows, new columns solved_label and priority added.


## 2. Bar chart — Crime counts by type

In [ ]:
ggplot(crimes, aes(x = type, fill = type)) +
  geom_bar(show.legend = FALSE) +
  coord_flip() +
  labs(title = "Crime Counts by Type", x = "", y = "Number of Incidents") +
  theme_minimal()
# Resulting plot:


![Bar counts by type](criminology_bar_type.png)

## 3. Histogram — Monthly incidents + threshold

In [ ]:
ggplot(crimes, aes(x = monthly_incidents)) +
  geom_histogram(bins = 12, fill = "#2E75B6", color = "white", alpha = 0.85) +
  geom_vline(xintercept = 15000, linetype = "dashed", color = "red", size = 1) +
  labs(title = "Distribution of Monthly Incidents",
       subtitle = "Red line = high-priority threshold (15,000)",
       x = "Monthly Incidents", y = "Count") +
  theme_minimal()


![Histogram monthly](criminology_hist_monthly.png)

## 4. Boxplot — Severity by crime type

In [ ]:
ggplot(crimes, aes(x = type, y = severity_score, fill = type)) +
  geom_boxplot(show.legend = FALSE, outlier.color = "red") +
  coord_flip() +
  labs(title = "Severity Score by Crime Type", x = "", y = "Severity Score (1–10)") +
  theme_minimal()


![Boxplot severity](criminology_box_severity.png)

## 5. Scatter — Monthly incidents vs media mentions

In [ ]:
ggplot(crimes, aes(x = monthly_incidents, y = media_mentions, color = solved_label)) +
  geom_point(size = 3, alpha = 0.8) +
  geom_smooth(method = "lm", se = FALSE, color = "gray", linetype = "dashed") +
  labs(title = "Monthly Incidents vs Media Mentions",
       x = "Monthly Incidents", y = "Media Mentions", color = "Status") +
  theme_minimal()


![Scatter media](criminology_scatter_media.png)

## 6. Priority highlight bar

In [ ]:
ggplot(crimes, aes(x = priority, fill = priority)) +
  geom_bar(show.legend = FALSE) +
  labs(title = "High-Priority vs Other Cases",
       subtitle = "High Priority = monthly > 15k and type ≠ Theft",
       x = "", y = "Count") +
  theme_minimal()


![Priority bar](criminology_priority_bar.png)

## 7. Mean monthly incidents by type

In [ ]:
means <- crimes %>%
  group_by(type) %>%
  summarise(mean_monthly = mean(monthly_incidents), .groups = "drop")

ggplot(means, aes(x = type, y = mean_monthly, fill = type)) +
  geom_col(show.legend = FALSE) +
  coord_flip() +
  labs(title = "Average Monthly Incidents by Crime Type", x = "", y = "Mean Monthly Incidents") +
  theme_minimal()


![Mean monthly](criminology_mean_monthly.png)

## Alternate Approaches

In [ ]:
# Count then geom_col + reorder
crimes %>%
  count(type) %>%
  ggplot(aes(x = reorder(type, n), y = n, fill = type)) +
  geom_col(show.legend = FALSE) +
  coord_flip() +
  labs(title = "Crime Counts (reordered)", x = "", y = "n") +
  theme_minimal()

# Facet example (by solved status)
ggplot(crimes, aes(x = monthly_incidents, fill = type)) +
  geom_histogram(bins = 10) +
  facet_wrap(~ solved_label) +
  theme_minimal()


## More Practice — Solutions

In [ ]:
# 1. Violin
ggplot(crimes, aes(x = type, y = severity_score, fill = type)) +
  geom_violin(show.legend = FALSE) +
  coord_flip() +
  theme_minimal()

# 2. Bar with count labels
crimes %>%
  count(type) %>%
  ggplot(aes(x = reorder(type, n), y = n, fill = type)) +
  geom_col(show.legend = FALSE) +
  geom_text(aes(label = n), hjust = -0.2, size = 3.5) +
  coord_flip() +
  theme_minimal()

# 3. Bubble
ggplot(crimes, aes(x = monthly_incidents, y = severity_score, size = media_mentions, color = type)) +
  geom_point(alpha = 0.7) +
  scale_size_continuous(range = c(2, 12)) +
  theme_minimal()


## Simulation / What-if Visuals

In [ ]:
# Change threshold and re-plot
threshold <- 18000
crimes2 <- crimes %>%
  mutate(priority = ifelse(monthly_incidents > threshold & type != "Theft",
                           "High Priority", "Other"))

ggplot(crimes2, aes(x = priority, fill = priority)) +
  geom_bar(show.legend = FALSE) +
  labs(title = paste("Priority cases at threshold =", threshold),
       x = "", y = "Count") +
  theme_minimal()

# Updated histogram line
ggplot(crimes, aes(x = monthly_incidents)) +
  geom_histogram(bins = 12, fill = "#2E75B6", color = "white") +
  geom_vline(xintercept = threshold, linetype = "dashed", color = "darkred") +
  labs(title = "Monthly incidents with new threshold",
       subtitle = paste("Threshold now", threshold)) +
  theme_minimal()


## Key Visual Takeaways

- Assault is the most frequent type; Homicide and Drug Offense show highest average severity.
- The 15 000 monthly-incident line cleanly separates a high-volume tail that decision makers should prioritize.
- Media attention rises with volume; unsolved high-volume cases stand out on the scatter.
- Changing the threshold in the simulation cell immediately updates the visual story for resource discussions.
